In [4]:
import pandas as pd
from linearmodels.panel import PanelOLS
from pathlib import Path

# 1. Cargar la base de datos
DIRECTORIO_BASE = Path.cwd().parent 
PATH_PROCESSED = DIRECTORIO_BASE / "data" / "processed"
df_panel = pd.read_excel(PATH_PROCESSED / "panel_econometrico_base.xlsx")

# 2. Configuración obligatoria para Panel Data (MultiIndex)
df_panel = df_panel.set_index(['departamento', 'año'])

# 3. CORRECCIÓN: Mapear explícitamente y ordenar categorías
# Creamos el diccionario asegurando que los números se vuelvan texto
diccionario = {
    0: 'Diversificado/Agro',
    1: 'Servicios/Manufactura',
    2: 'Minero/Enclave',
    3: 'Minero/Diversificado'
}
# Ojo: Asegúrate de que la columna se llame 'Cluster' o 'Cluster_K4' según tu Excel
df_panel['Nombre_Cluster'] = df_panel['Cluster'].map(diccionario)

# Ahora sí le aplicamos el ordenamiento categórico a la columna de texto
df_panel['Nombre_Cluster'] = pd.Categorical(
    df_panel['Nombre_Cluster'], 
    categories=['Diversificado/Agro', 'Servicios/Manufactura', 'Minero/Enclave', 'Minero/Diversificado'],
    ordered=True
)

# 4. Construir la regresión
# var_pib: dependiente
# 1: intercepto
# COVID: El efecto sobre el grupo base (Agro)
# COVID:Nombre_Cluster: Cuánto PEOR o MEJOR le fue a los otros vs Agro
# EntityEffects: Efectos Fijos
formula = 'var_pib ~ 1 + COVID + COVID:Nombre_Cluster + EntityEffects'

# 5. Entrenar el modelo
modelo = PanelOLS.from_formula(formula, data=df_panel)
resultados = modelo.fit(cov_type='robust')

print(resultados)

                          PanelOLS Estimation Summary                           
Dep. Variable:                var_pib   R-squared:                        0.1390
Estimator:                   PanelOLS   R-squared (Between):             -0.0406
No. Observations:                 428   R-squared (Within):               0.1390
Date:                Sat, Sep 05 2026   R-squared (Overall):              0.1312
Time:                        15:47:37   Log-likelihood                   -1521.2
Cov. Estimator:                Robust                                           
                                        F-statistic:                      16.140
Entities:                          24   P-value                           0.0000
Avg Obs:                       17.833   Distribution:                   F(4,400)
Min Obs:                       16.000                                           
Max Obs:                       18.000   F-statistic (robust):             70.550
                            